# Notebook 02: Document Conversion

## KLEAN Chatbot — Restaurant Information Retrieval & Recommendation

### Overview
Information Retrieval algorithms (such as Inverted Index, TF-IDF, BM25, and Vector Embeddings) require text documents as input. In this notebook, we convert structured JSON restaurant records into searchable documents using `DocumentConverter` from `src.utils.document_converter`.

**Core Objectives:**
1. Convert structured records into searchable textual strings.
2. Ensure null or missing values are cleanly excluded.
3. Maintain explicit bidirectional mapping between `doc_id` and the original structured restaurant data.
4. Generate a clean corpus dictionary `{doc_id: text}` for subsequent IR stages.

In [1]:
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.data_loader import DataLoader
from src.utils.document_converter import DocumentConverter, SearchDocument

print(f"Project root resolved: {PROJECT_ROOT}")

Project root resolved: D:\Year4\Web_Retrive\Project


## 1. Load Records and Initialize DocumentConverter

In [2]:
loader = DataLoader(PROJECT_ROOT / "dataset" / "restaurants.json")
records = loader.load()

converter = DocumentConverter()
documents = converter.convert_all(records)

print(f"Total converted documents: {len(documents)}")

Total converted documents: 698


## 2. Inspect Converted SearchDocuments

Let's look at the generated textual documents to ensure readability and completeness.

In [3]:
for doc in documents[:5]:
    print(f"--- Document ID: {doc.doc_id} ---")
    print(f"Text: {doc.text}")
    print(f"Metadata: {doc.metadata}\n")

--- Document ID: 1 ---
Text: Name: 13Eo Breakfast | Category: Breakfast And Brunch Restaurant | Sub Category: Vietnamese Restaurant, Tea Room | Address: 13Eo Breakfast, ភ្នំពេញ | Postal Code: 121201
Metadata: {'name': '13Eo Breakfast', 'category': 'Breakfast And Brunch Restaurant', 'sub_category': ['Vietnamese Restaurant', 'Tea Room'], 'address': '13Eo Breakfast, ភ្នំពេញ', 'postal_code': '121201', 'latitude': 11.53061580657959, 'longitude': 104.93585205078125, 'phone': '+85511256604', 'website': None}

--- Document ID: 2 ---
Text: Name: 258 Local Restaurant | Category: Cambodian Restaurant | Sub Category: Restaurant | Address: Long Nget St. (258), ភ្នំពេញ | Postal Code: 120207
Metadata: {'name': '258 Local Restaurant', 'category': 'Cambodian Restaurant', 'sub_category': ['Restaurant'], 'address': 'Long Nget St. (258), ភ្នំពេញ', 'postal_code': '120207', 'latitude': 11.559806823730469, 'longitude': 104.93441772460938, 'phone': None, 'website': None}

--- Document ID: 3 ---
Text: Name: 27

## 3. Verify Null Value Handling

Compare a restaurant record with null attributes (e.g. record 8, which has no website or email) to verify that null fields are omitted from the text.

In [4]:
rec_8 = loader.get_by_id(8)
doc_8 = converter.convert_record(rec_8)

print("Original Record 8 (website & email null):", rec_8.get("website"), rec_8.get("email"))
print("Generated Document 8 text:")
print(doc_8.text)

Original Record 8 (website & email null): None None
Generated Document 8 text:
Name: 560 Restaurants and KTV | Category: Asian Restaurant | Sub Category: Karaoke, Restaurant | Address: 2F. 53, Street 560, Phum 11, ភ្នំពេញ | Postal Code: 12151


## 4. Document ID Mapping and Corpus Generation

The IR engine needs to map search results back to the original restaurant records.

In [5]:
# Retrieve original record via converter
retrieved_record = converter.get_record(doc_id=2)
print(f"Mapped Record for ID 2: {retrieved_record['name']} | Phone: {retrieved_record['phone']}")

# Extract corpus for indexing
corpus = converter.get_corpus()
print(f"Corpus size: {len(corpus)} entries")
for doc_id, text in list(corpus.items())[:3]:
    print(f"[{doc_id}] -> {text[:80]}...")

Mapped Record for ID 2: 258 Local Restaurant | Phone: None
Corpus size: 698 entries
[1] -> Name: 13Eo Breakfast | Category: Breakfast And Brunch Restaurant | Sub Category:...
[2] -> Name: 258 Local Restaurant | Category: Cambodian Restaurant | Sub Category: Rest...
[3] -> Name: 27 Wine and Spirits - Wholesale | Category: Asian Restaurant | Sub Categor...


## 5. Summary & Next Steps
- Every restaurant is converted into a standardized text document preserving key searchable terms (name, category, subcategory, address, postal code).
- Mapping between `doc_id` and structured restaurant attributes is preserved.
- In **Phase 2**, we will preprocess these texts (tokenization, lowercasing, normalization) and construct our **Inverted Index**.